In [1]:
using LinearAlgebra
using Statistics
using PlotlyJS

WebIO._IJuliaInit()

In [2]:
c = [-3.0, -2.0]

raw_constraints = [
    ([1.0, 2.0], "<=", 7.0),
    ([2.0, 1.0], "<=", 8.0),
    ([0.0, 1.0], "<=", 3.0)
]

function normalize_constraints(raw)
    cons = Vector{Tuple{Vector{Float64}, Float64}}()
    for (a, sense, b) in raw
        aa = Float64.(a)
        bb = Float64(b)
        if sense == "<="
            push!(cons, (aa, bb))
        elseif sense == ">="
            push!(cons, (-aa, -bb))
        else
            error("Unknown sense: $sense")
        end
    end
    return cons
end

constraints = normalize_constraints(raw_constraints)

push!(constraints, ([-1.0,  0.0], 0.0))
push!(constraints, ([ 0.0, -1.0], 0.0))

constraints


5-element Vector{Tuple{Vector{Float64}, Float64}}:
 ([1.0, 2.0], 7.0)
 ([2.0, 1.0], 8.0)
 ([0.0, 1.0], 3.0)
 ([-1.0, 0.0], 0.0)
 ([0.0, -1.0], 0.0)

In [3]:
const TOL = 1e-9

f(x, c) = dot(c, x)
grad(c) = c
antigrad(c) = -c

function feasible(x, cons; tol=TOL)
    for (a, b) in cons
        if dot(a, x) - b > tol
            return false
        end
    end
    return true
end

function line_intersection(a1, b1, a2, b2; tol=TOL)
    # решаем [a1'; a2'] x = [b1; b2]
    A = [a1[1] a1[2]; a2[1] a2[2]]
    detA = det(A)
    if abs(detA) < tol
        return nothing
    end
    x = A \ [b1; b2]
    return vec(x)
end

function unique_points(points; tol=1e-7)
    uniq = Vector{Vector{Float64}}()
    for p in points
        ok = true
        for q in uniq
            if norm(p .- q) <= tol
                ok = false
                break
            end
        end
        ok && push!(uniq, p)
    end
    return uniq
end

function convex_hull(points)
    # Andrew monotonic chain, возвращает вершины hull по кругу (без повтора первой)
    pts = sort(points, by = p -> (p[1], p[2]))
    if length(pts) <= 1
        return pts
    end
    cross(o, a, b) = (a[1]-o[1])*(b[2]-o[2]) - (a[2]-o[2])*(b[1]-o[1])
    lower = Vector{Vector{Float64}}()
    for p in pts
        while length(lower) >= 2 && cross(lower[end-1], lower[end], p) <= 1e-12
            pop!(lower)
        end
        push!(lower, p)
    end
    upper = Vector{Vector{Float64}}()
    for p in reverse(pts)
        while length(upper) >= 2 && cross(upper[end-1], upper[end], p) <= 1e-12
            pop!(upper)
        end
        push!(upper, p)
    end
    hull = vcat(lower[1:end-1], upper[1:end-1])
    return hull
end

function all_intersections(cons)
    lines = [(a, b) for (a, b) in cons]
    pts = Vector{Vector{Float64}}()
    for i in 1:length(lines)
        for j in i+1:length(lines)
            a1, b1 = lines[i]
            a2, b2 = lines[j]
            x = line_intersection(a1, b1, a2, b2)
            x === nothing && continue
            push!(pts, x)
        end
    end
    return unique_points(pts)
end

function objective_unbounded_below(c, cons; samples_per_sector=1, tol=1e-12)
    # Проверяем, существует ли допустимое направление d (|d|=1), такое что:
    #   a⋅d <= 0 для всех ограничений (движение остаётся в X)
    #   c⋅d < 0  => f(x+td) = f(x) + t c⋅d -> -∞
    thetas = Float64[]
    for (a, _) in cons
        a1, a2 = a
        if abs(a1) > tol || abs(a2) > tol
            θ = atan(-a1, a2)  # решение a1 cosθ + a2 sinθ = 0
            push!(thetas, θ)
            push!(thetas, θ + pi)
        end
    end
    push!(thetas, -pi)
    push!(thetas, pi)
    sort!(thetas)

    u = Float64[]
    for θ in thetas
        if isempty(u) || abs(θ - u[end]) > 1e-8
            push!(u, θ)
        end
    end
    thetas = u

    function dir_ok(θ)
        d = [cos(θ), sin(θ)]
        for (a, _) in cons
            if dot(a, d) > 1e-10
                return false
            end
        end
        return dot(c, d) < -1e-10
    end

    # Проверяем середины между критическими углами
    for k in 1:(length(thetas)-1)
        θm = 0.5*(thetas[k] + thetas[k+1])
        if dir_ok(θm)
            return true
        end
    end
    # И сами критические точки тоже
    for θ in thetas
        if dir_ok(θ)
            return true
        end
    end
    return false
end

function solve_lp_graphical(c, cons; tol=TOL)
    pts = all_intersections(cons)
    feas = [p for p in pts if feasible(p, cons; tol=tol)]
    feas = unique_points(feas)

    if isempty(feas)
        return (status=:infeasible, feasible_points=feas, hull=Vector{Vector{Float64}}(), opt=nothing)
    end

    if objective_unbounded_below(c, cons)
        hull = convex_hull(feas)
        return (status=:unbounded, feasible_points=feas, hull=hull, opt=nothing)
    end

    vals = [f(p, c) for p in feas]
    fmin = minimum(vals)
    idx = findall(v -> abs(v - fmin) <= 1e-7, vals)
    minimizers = feas[idx]

    hull = convex_hull(feas)

    if length(minimizers) == 1
        return (status=:optimal_point, feasible_points=feas, hull=hull, opt=(type=:point, x=minimizers[1], f=fmin))
    else
        best = (0.0, minimizers[1], minimizers[2])
        for i in 1:length(minimizers)
            for j in i+1:length(minimizers)
                d = norm(minimizers[i] .- minimizers[j])
                if d > best[1]
                    best = (d, minimizers[i], minimizers[j])
                end
            end
        end
        xA, xB = best[2], best[3]
        if best[1] <= 1e-7
            return (status=:optimal_point, feasible_points=feas, hull=hull, opt=(type=:point, x=minimizers[1], f=fmin))
        end
        return (status=:optimal_segment, feasible_points=feas, hull=hull, opt=(type=:segment, A=xA, B=xB, f=fmin))
    end
end

res = solve_lp_graphical(c, constraints)
res.status, res.opt

(:optimal_point, (type = :point, x = [3.0, 2.0], f = -13.0))

In [4]:
function sort_by_direction(points, dir)
    scores = [dot(dir, p) for p in points]
    order = sortperm(scores; rev=true)
    return points[order], scores[order]
end

dir = antigrad(c)
sorted_pts, scores = sort_by_direction(res.hull, dir)

hcat([p[1] for p in sorted_pts], [p[2] for p in sorted_pts], scores)

5×3 Matrix{Float64}:
  3.0   2.0  13.0
  4.0  -0.0  12.0
  1.0   3.0   9.0
 -0.0   3.0   6.0
 -0.0  -0.0   0.0

## 4) Визуализация (интерактивная 3D)

- Внизу (плоскость z=0z=0) рисуем область XX и ограничения.
- На плоскости z=f(x)z=f(x) рисуем «крышку» по вершинам области.
- Линии уровня: c⊤x=constc^\top x = \text{const} (в плоскости z=0z=0).
- Стрелка антиградиента: −c-c.

Как читать вашу картинку:
- **`X boundary` (толстая линия)**: граница допустимой области XX в плоскости x1,x2x_1,x_2 (внизу, z=0z=0).
- **`feasible vertices` (точки)**: все найденные точки пересечения, которые удовлетворяют всем неравенствам (включая x1≥0,x2≥0x_1\ge0, x_2\ge0).
- **`a⋅x=b` (пунктир)**: граничные прямые ограничений.
- **`level` (тонкие линии)**: линии уровня c⊤x=constc^\top x = \text{const} в плоскости z=0z=0.
- **`-∇f direction` (стрелка)**: направление антиградиента −c-c в плоскости z=0z=0.
- **`opt / opt segment` (красное)**: найденный оптимум (точка или отрезок) в плоскости z=0z=0.
- **`f(x) over X` (полупрозрачная «крышка»)**: значение \(f(x)\) над вершинами области (ось \(f(x)\) направлена вверх).


In [5]:
function plot_solution(c, cons, res; level_count=6, title_suffix="")
    # 2D-картинка в плоскости (x1, x2). Ось "вверх" не используем.
    feas = res.feasible_points
    hull = res.hull
    dir = antigrad(c)

    function default_window(cons)
        # Пытаемся оценить окно по пересечениям граничных прямых и осям.
        pts = all_intersections(cons)
        # Добавим пересечения с осями x1=0 и x2=0
        for (a, b) in cons
            a1, a2 = a
            if abs(a2) > 1e-12
                y = b / a2
                push!(pts, [0.0, y])
            end
            if abs(a1) > 1e-12
                x = b / a1
                push!(pts, [x, 0.0])
            end
        end
        pts = unique_points(pts)
        if isempty(pts)
            return (-1.0, 5.0, -1.0, 5.0)
        end
        xs = [p[1] for p in pts]; ys = [p[2] for p in pts]
        xmin, xmax = minimum(xs), maximum(xs)
        ymin, ymax = minimum(ys), maximum(ys)
        span = max(xmax-xmin, ymax-ymin)
        span = max(span, 2.0)
        pad = 0.35 * span
        return (xmin - pad, xmax + pad, ymin - pad, ymax + pad)
    end

    xmin, xmax, ymin, ymax = default_window(cons)

    # Быстрое отображение области X для unbounded/infeasible: пересекаем полуплоскости с прямоугольным окном.
    inside(p, a, b) = dot(a, p) <= b + 1e-9
    function intersect_seg_with_line(S, E, a, b)
        d = E .- S
        denom = dot(a, d)
        if abs(denom) < 1e-12
            return S
        end
        t = (b - dot(a, S)) / denom
        return S .+ t .* d
    end

    function clip_polygon(poly, a, b)
        if isempty(poly)
            return poly
        end
        out = Vector{Vector{Float64}}()
        S = poly[end]
        S_in = inside(S, a, b)
        for E in poly
            E_in = inside(E, a, b)
            if E_in
                if !S_in
                    push!(out, intersect_seg_with_line(S, E, a, b))
                end
                push!(out, E)
            elseif S_in
                push!(out, intersect_seg_with_line(S, E, a, b))
            end
            S = E
            S_in = E_in
        end
        return out
    end

    window_fill = nothing
    if res.status in (:unbounded, :infeasible)
        rect = [[xmin, ymin], [xmax, ymin], [xmax, ymax], [xmin, ymax]]
        poly = rect
        for (a, b) in cons
            poly = clip_polygon(poly, a, b)
            isempty(poly) && break
        end
        if length(poly) >= 3
            px = [p[1] for p in poly]; py = [p[2] for p in poly]
            px = vcat(px, px[1]); py = vcat(py, py[1])
            window_fill = scatter(
                x=px, y=py,
                mode="lines",
                line=attr(width=2, color="rgba(31,119,180,0.8)"),
                fill="toself",
                fillcolor="rgba(31,119,180,0.15)",
                name="X (в окне)"
            )
        end
    end

    # 1) Точки-вершины (если есть)
    pts_trace = isempty(feas) ? nothing : scatter(
        x=[p[1] for p in feas],
        y=[p[2] for p in feas],
        mode="markers",
        marker=attr(size=8, color="rgba(0,0,0,0.65)"),
        name="вершины (пересечения)"
    )

    # 2) Граница области X (только если есть замкнутый hull)
    hull_trace = nothing
    if length(hull) >= 3
        hx = [p[1] for p in hull]; hy = [p[2] for p in hull]
        hx = vcat(hx, hx[1]); hy = vcat(hy, hy[1])
        hull_trace = scatter(
            x=hx, y=hy,
            mode="lines",
            line=attr(width=5, color="rgba(31,119,180,1.0)"),
            fill="toself",
            fillcolor="rgba(31,119,180,0.10)",
            name="граница X (если ограничена)"
        )
    end

    # 3) Граничные прямые ограничений a⋅x=b
    line_traces = PlotlyJS.GenericTrace[]
    for (a, b) in cons
        a1, a2 = a
        if abs(a2) > 1e-12
            x1s = [xmin, xmax]
            ys2 = [(b - a1*x)/a2 for x in x1s]
            push!(line_traces, scatter(x=x1s, y=ys2, mode="lines", line=attr(width=2, dash="dot", color="rgba(120,120,120,0.8)"), name="a⋅x=b"))
        elseif abs(a1) > 1e-12
            x = b/a1
            push!(line_traces, scatter(x=[x, x], y=[ymin, ymax], mode="lines", line=attr(width=2, dash="dot", color="rgba(120,120,120,0.8)"), name="a⋅x=b"))
        end
    end

    # 4) Линии уровня c⋅x = const
    # Для unbounded/малого числа вершин vmin≈vmax и уровни "слипаются".
    # Поэтому берём диапазон уровней по углам окна просмотра.
    corners = [[xmin, ymin], [xmin, ymax], [xmax, ymin], [xmax, ymax]]
    corner_vals = [f(p, c) for p in corners]
    vmin, vmax = minimum(corner_vals), maximum(corner_vals)
    if abs(vmax - vmin) < 1e-9
        mid = 0.5*(vmin+vmax)
        vmin = mid - 1.0
        vmax = mid + 1.0
    end
    levels = range(vmin, vmax; length=level_count)
    level_traces = PlotlyJS.GenericTrace[]
    for L in levels
        c1, c2 = c
        if abs(c2) > 1e-12
            x1s = [xmin, xmax]
            ys2 = [(L - c1*x)/c2 for x in x1s]
            push!(level_traces, scatter(x=x1s, y=ys2, mode="lines", line=attr(width=2, color="rgba(255,127,14,0.9)"), name="линия уровня"))
        elseif abs(c1) > 1e-12
            x = L/c1
            push!(level_traces, scatter(x=[x, x], y=[ymin, ymax], mode="lines", line=attr(width=2, color="rgba(255,127,14,0.9)"), name="линия уровня"))
        end
    end

    # 5) Оптимум (точка/отрезок) в плоскости
    opt_traces = PlotlyJS.GenericTrace[]
    if res.status == :optimal_point
        x = res.opt.x
        push!(opt_traces, scatter(x=[x[1]], y=[x[2]], mode="markers", marker=attr(size=12, color="red"), name="оптимум"))
    elseif res.status == :optimal_segment
        A = res.opt.A; B = res.opt.B
        push!(opt_traces, scatter(x=[A[1], B[1]], y=[A[2], B[2]], mode="lines+markers", line=attr(width=6, color="red"), marker=attr(size=10, color="red"), name="оптимальный отрезок"))
    end

    # 6) Стрелка антиградиента (-c) как annotation
    center = length(hull) >= 1 ? [mean([p[1] for p in hull]), mean([p[2] for p in hull])] : [0.5*(xmin+xmax), 0.5*(ymin+ymax)]
    d = dir / (norm(dir) + 1e-12)
    scale = 0.35 * max(xmax-xmin, ymax-ymin)
    tip = center .+ d .* (0.35*scale)
    ann = [attr(
        x=tip[1], y=tip[2], ax=center[1], ay=center[2],
        xref="x", yref="y", axref="x", ayref="y",
        showarrow=true, arrowhead=3, arrowsize=1.2, arrowwidth=3, arrowcolor="black",
        text="-∇f", font=attr(color="black")
    )]

    traces = PlotlyJS.GenericTrace[]
    window_fill !== nothing && push!(traces, window_fill)
    hull_trace !== nothing && push!(traces, hull_trace)
    pts_trace !== nothing && push!(traces, pts_trace)
    append!(traces, line_traces)
    append!(traces, level_traces)
    append!(traces, opt_traces)
    if res.status == :infeasible
        push!(ann, attr(x=0.5*(xmin+xmax), y=0.5*(ymin+ymax), xref="x", yref="y", showarrow=false, text="X = ∅", font=attr(color="red", size=18)))
    end
    p = PlotlyJS.Plot(traces)
    relayout!(p,
        title="ЛП в R2 (графический метод)" * title_suffix,
        xaxis=attr(title="x1", range=[xmin, xmax], zeroline=true),
        yaxis=attr(title="x2", range=[ymin, ymax], zeroline=true, scaleanchor="x", scaleratio=1),
        legend=attr(orientation="h"),
        annotations=ann
    )
    return p
end


plot_solution (generic function with 1 method)

In [6]:
p = plot_solution(c, constraints, res)
p


data: [
  "scatter with fields fill, fillcolor, line, mode, name, type, x, and y",
  "scatter with fields marker, mode, name, type, x, and y",
  "scatter with fields line, mode, name, type, x, and y",
  "scatter with fields line, mode, name, type, x, and y",
  "scatter with fields line, mode, name, type, x, and y",
  "scatter with fields line, mode, name, type, x, and y",
  "scatter with fields line, mode, name, type, x, and y",
  "scatter with fields line, mode, name, type, x, and y",
  "scatter with fields line, mode, name, type, x, and y",
  "scatter with fields line, mode, name, type, x, and y",
  "scatter with fields line, mode, name, type, x, and y",
  "scatter with fields line, mode, name, type, x, and y",
  "scatter with fields line, mode, name, type, x, and y",
  "scatter with fields marker, mode, name, type, x, and y"
]

layout: "layout with fields annotations, legend, margin, template, title, xaxis, and yaxis"

In [7]:
res.status

if res.status == :optimal_point
    x = res.opt.x
    println("x* = ", x, "  f(x*) = ", res.opt.f)
elseif res.status == :optimal_segment
    println("Optimal segment endpoints:")
    println("A = ", res.opt.A, "  B = ", res.opt.B, "  f = ", res.opt.f)
elseif res.status == :unbounded
    println("Objective is unbounded below (f -> -∞)")
elseif res.status == :infeasible
    println("No feasible points")
else
    println("Status: ", res.status)
end


x* = [3.0, 2.0]  f(x*) = -13.0


In [8]:
examples = Dict{String, Any}()
case_ru = Dict(
    "infeasible" => "Решения не существует (X пуста)",
    "unbounded" => "Решение уходит в бесконечность (f → -∞)",
    "optimal_point" => "Решение — точка",
    "optimal_segment" => "Решение — отрезок"
)

function objective_str(c)
    c1, c2 = c
    return "f(x) = $(c1)*x1 + $(c2)*x2"
end

function print_constraints(cons)
    println("Ограничения (в виде a⋅x ≤ b):")
    for (a, b) in cons
        println("  $(a[1])*x1 + $(a[2])*x2 ≤ $(b)")
    end
end

function print_solution(res, c, cons)
    println(objective_str(c))
    print_constraints(cons)
    if res.status == :optimal_point
        x = res.opt.x
        println("x* = ", x)
        println("f(x*) = ", f(x, c))
    elseif res.status == :optimal_segment
        A, B = res.opt.A, res.opt.B
        println("Оптимальный отрезок: A = ", A, "   B = ", B)
        println("f(A) = ", f(A, c), "   f(B) = ", f(B, c))
    elseif res.status == :unbounded
        println("f(x) не ограничена снизу на X (f → -∞)")
    elseif res.status == :infeasible
        println("Допустимая область X пуста")
    else
        println("status = ", res.status)
    end
end

# 1) Решения не существует: x1 >= 1 и x1 <= 0 одновременно (только <=)
examples["infeasible"] = (
    c = [1.0, 1.0],
    raw = [
        ([1.0, 0.0], "<=", 0.0),
        ([-1.0, 0.0], "<=", -1.0)
    ]
)

# 2) Уходит в бесконечность: X неограничено и есть допустимое направление убывания f
# Ограничения: x1 - x2 <= 0, x1>=0, x2>=0 => можно идти вдоль d=(0,1), а при c=[1,-1] получаем c⋅d=-1 < 0
examples["unbounded"] = (
    c = [1.0, -1.0],
    raw = [
        ([1.0, -1.0], "<=", 0.0)
    ]
)

# 3) Оптимум-точка (классический ограниченный многоугольник)
examples["optimal_point"] = (
    c = [-3.0, -2.0],
    raw = [
        ([1.0, 2.0], "<=", 7.0),
        ([2.0, 1.0], "<=", 8.0),
        ([0.0, 1.0], "<=", 3.0)
    ]
)

# 4) Оптимум-отрезок: минимизируем x1 при 0<=x1<=1, 0<=x2<=1
# Тогда минимум f=x1 достигается на всём отрезке x1=0, x2∈[0,1]
examples["optimal_segment"] = (
    c = [1.0, 0.0],
    raw = [
        ([1.0, 0.0], "<=", 1.0),
        ([0.0, 1.0], "<=", 1.0)
    ]
)

(c = [1.0, 0.0], raw = [([1.0, 0.0], "<=", 1.0), ([0.0, 1.0], "<=", 1.0)])

In [9]:
for nm in ["infeasible", "unbounded", "optimal_point", "optimal_segment"]
    ex = examples[nm]
    c_ex = ex.c
    constraints_ex = normalize_constraints(ex.raw)
    # Неявные ограничения первой четверти
    push!(constraints_ex, ([-1.0,  0.0], 0.0))
    push!(constraints_ex, ([ 0.0, -1.0], 0.0))

    res_ex = solve_lp_graphical(c_ex, constraints_ex)
    println("\n===== ", case_ru[nm], " =====")
    println("status = ", res_ex.status)
    print_solution(res_ex, c_ex, constraints_ex)
    display(plot_solution(c_ex, constraints_ex, res_ex; title_suffix=" — " * case_ru[nm]))
end



===== Решения не существует (X пуста) =====
status = infeasible
f(x) = 1.0*x1 + 1.0*x2
Ограничения (в виде a⋅x ≤ b):
  1.0*x1 + 0.0*x2 ≤ 0.0
  -1.0*x1 + 0.0*x2 ≤ -1.0
  -1.0*x1 + 0.0*x2 ≤ 0.0
  0.0*x1 + -1.0*x2 ≤ 0.0
Допустимая область X пуста


data: [
  "scatter with fields line, mode, name, type, x, and y",
  "scatter with fields line, mode, name, type, x, and y",
  "scatter with fields line, mode, name, type, x, and y",
  "scatter with fields line, mode, name, type, x, and y",
  "scatter with fields line, mode, name, type, x, and y",
  "scatter with fields line, mode, name, type, x, and y",
  "scatter with fields line, mode, name, type, x, and y",
  "scatter with fields line, mode, name, type, x, and y",
  "scatter with fields line, mode, name, type, x, and y",
  "scatter with fields line, mode, name, type, x, and y"
]

layout: "layout with fields annotations, legend, margin, template, title, xaxis, and yaxis"


===== Решение уходит в бесконечность (f → -∞) =====
status = unbounded
f(x) = 1.0*x1 + -1.0*x2
Ограничения (в виде a⋅x ≤ b):
  1.0*x1 + -1.0*x2 ≤ 0.0
  -1.0*x1 + 0.0*x2 ≤ 0.0
  0.0*x1 + -1.0*x2 ≤ 0.0
f(x) не ограничена снизу на X (f → -∞)


data: [
  "scatter with fields fill, fillcolor, line, mode, name, type, x, and y",
  "scatter with fields marker, mode, name, type, x, and y",
  "scatter with fields line, mode, name, type, x, and y",
  "scatter with fields line, mode, name, type, x, and y",
  "scatter with fields line, mode, name, type, x, and y",
  "scatter with fields line, mode, name, type, x, and y",
  "scatter with fields line, mode, name, type, x, and y",
  "scatter with fields line, mode, name, type, x, and y",
  "scatter with fields line, mode, name, type, x, and y",
  "scatter with fields line, mode, name, type, x, and y",
  "scatter with fields line, mode, name, type, x, and y"
]

layout: "layout with fields annotations, legend, margin, template, title, xaxis, and yaxis"


===== Решение — точка =====
status = optimal_point
f(x) = -3.0*x1 + -2.0*x2
Ограничения (в виде a⋅x ≤ b):
  1.0*x1 + 2.0*x2 ≤ 7.0
  2.0*x1 + 1.0*x2 ≤ 8.0
  0.0*x1 + 1.0*x2 ≤ 3.0
  -1.0*x1 + 0.0*x2 ≤ 0.0
  0.0*x1 + -1.0*x2 ≤ 0.0
x* = [3.0, 2.0]
f(x*) = -13.0


data: [
  "scatter with fields fill, fillcolor, line, mode, name, type, x, and y",
  "scatter with fields marker, mode, name, type, x, and y",
  "scatter with fields line, mode, name, type, x, and y",
  "scatter with fields line, mode, name, type, x, and y",
  "scatter with fields line, mode, name, type, x, and y",
  "scatter with fields line, mode, name, type, x, and y",
  "scatter with fields line, mode, name, type, x, and y",
  "scatter with fields line, mode, name, type, x, and y",
  "scatter with fields line, mode, name, type, x, and y",
  "scatter with fields line, mode, name, type, x, and y",
  "scatter with fields line, mode, name, type, x, and y",
  "scatter with fields line, mode, name, type, x, and y",
  "scatter with fields line, mode, name, type, x, and y",
  "scatter with fields marker, mode, name, type, x, and y"
]

layout: "layout with fields annotations, legend, margin, template, title, xaxis, and yaxis"


===== Решение — отрезок =====
status = optimal_segment
f(x) = 1.0*x1 + 0.0*x2
Ограничения (в виде a⋅x ≤ b):
  1.0*x1 + 0.0*x2 ≤ 1.0
  0.0*x1 + 1.0*x2 ≤ 1.0
  -1.0*x1 + 0.0*x2 ≤ 0.0
  0.0*x1 + -1.0*x2 ≤ 0.0
Оптимальный отрезок: A = [-0.0, 1.0]   B = [-0.0, -0.0]
f(A) = 0.0   f(B) = 0.0


data: [
  "scatter with fields fill, fillcolor, line, mode, name, type, x, and y",
  "scatter with fields marker, mode, name, type, x, and y",
  "scatter with fields line, mode, name, type, x, and y",
  "scatter with fields line, mode, name, type, x, and y",
  "scatter with fields line, mode, name, type, x, and y",
  "scatter with fields line, mode, name, type, x, and y",
  "scatter with fields line, mode, name, type, x, and y",
  "scatter with fields line, mode, name, type, x, and y",
  "scatter with fields line, mode, name, type, x, and y",
  "scatter with fields line, mode, name, type, x, and y",
  "scatter with fields line, mode, name, type, x, and y",
  "scatter with fields line, mode, name, type, x, and y",
  "scatter with fields line, marker, mode, name, type, x, and y"
]

layout: "layout with fields annotations, legend, margin, template, title, xaxis, and yaxis"